<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Box Plots**


Estimated time needed: **45** minutes


In this lab, you will focus on the visualization of data. The dataset will be provided through an RDBMS, and you will need to use SQL queries to extract the required data.


## Objectives


In this lab you will perform the following:


-   Visualize the distribution of data.

-   Visualize the relationship between two features.

-   Visualize data composition and comparisons using box plots.


### Setup: Connecting to the Database


#### 1. Download the Database File


In [1]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/QR9YeprUYhOoLafzlLspAw/survey-results-public.sqlite

--2026-06-27 15:15:56--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/QR9YeprUYhOoLafzlLspAw/survey-results-public.sqlite
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
200 OKequest sent, awaiting response... 
Length: 211415040 (202M) [application/octet-stream]
Saving to: ‘survey-results-public.sqlite’

-public.sqlite       43%[=======>            ]  88.09M  19.4MB/s    eta 7s     ^C


#### 2. Connect to the Database


**Install the needed libraries**


In [2]:
!pip install pandas

In [3]:
!pip install matplotlib

In [4]:
!pip install seaborn 

In [5]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Connect to the SQLite database
conn = sqlite3.connect('survey-results-public.sqlite')


## Demo: Basic SQL Queries


#### Demo 1: Count the Number of Rows in the Table


In [6]:
QUERY = "SELECT COUNT(*) FROM main"
df = pd.read_sql_query(QUERY, conn)
print(df)


DatabaseError: Execution failed on sql 'SELECT COUNT(*) FROM main': database disk image is malformed

#### Demo 2: List All Tables


In [ ]:
QUERY = """
SELECT name as Table_Name 
FROM sqlite_master 
WHERE type = 'table'
"""
pd.read_sql_query(QUERY, conn)


#### Demo 3: Group Data by Age


In [ ]:
QUERY = """
SELECT Age, COUNT(*) as count 
FROM main 
GROUP BY Age 
ORDER BY Age
"""
df_age = pd.read_sql_query(QUERY, conn)
print(df_age)


## Visualizing Data


### Task 1: Visualizing the Distribution of Data


**1. Box Plot of `CompTotal` (Total Compensation)**


Use a box plot to analyze the distribution and outliers in total compensation.


In [ ]:
# your code goes here
QUERY1 = """
SELECT CompTotal FROM main 
WHERE CompTotal IS NOT NULL
"""
df1 = pd.read_sql_query(QUERY1, conn)
sns.boxplot(x=df1['CompTotal'], color='blue')
plt.title('Total Compensation')
plt.show()

**2. Box Plot of Age (converted to numeric values)**


Convert the `Age` column into numerical values and visualize the distribution.


In [ ]:
# your code goes here
QUERY2 = """
SELECT Age 
FROM main
WHERE Age IS NOT NULL
"""
df2 = pd.read_sql_query(QUERY2, conn)
sns.boxplot(x=df2['Age'], color='blue')
plt.title('Age')
plt.show()

### Task 2: Visualizing Relationships in Data


**1. Box Plot of `CompTotal` Grouped by Age Groups:**


Visualize the distribution of compensation across different age groups.


In [ ]:
# your code goes here
QUERY3 = """
SELECT CompTotal, Age 
FROM main
WHERE CompTotal IS NOT NULL AND Age IS NOT NULL
"""
df3 = pd.read_sql_query(QUERY3, conn)
sns.boxplot(data=df3, x='Age', y='CompTotal', color='blue')
plt.title('CompTotal Grouped by Age Groups')
plt.show()

**2. Box Plot of `CompTotal` Grouped by Job Satisfaction (`JobSatPoints_6`):**


Examine how compensation varies based on job satisfaction levels.


In [ ]:
# your code goes here
QUERY4 = """
SELECT CompTotal, JobSatPoints_6 
FROM main
WHERE CompTotal IS NOT NULL AND JobSatPoints_6 IS NOT NULL
"""
df4 = pd.read_sql_query(QUERY4, conn)
df4["CompTotal"] = pd.to_numeric(df4["CompTotal"], errors="coerce")
df4 = df4.dropna(subset=["CompTotal", "JobSatPoints_6"])
df4 = df4[df4["CompTotal"] < df4["CompTotal"].quantile(0.99)]
top10 = df4["JobSatPoints_6"].value_counts().nlargest(10).index
df4_top10 = df4[df4["JobSatPoints_6"].isin(top10)]
plt.figure(figsize=(10,6))
sns.boxplot(data=df4_top10, x="JobSatPoints_6", y="CompTotal", color='blue')
plt.title("CompTotal Grouped by Top 10 Job Satisfaction Levels")
plt.show()

### Task 3: Visualizing the Composition of Data


**1. Box Plot of `ConvertedCompYearly` for the Top 5 Developer Types:**


Analyze compensation across the top 5 developer roles.


In [ ]:
# your code goes here
QUERY5 = """
SELECT ConvertedCompYearly, DevType
FROM main
WHERE ConvertedCompYearly IS NOT NULL
  AND DevType IN (
      SELECT DevType
      FROM main
      GROUP BY DevType
      ORDER BY COUNT(*) DESC
      LIMIT 5
  )
"""
df5 = pd.read_sql_query(QUERY5, conn)
df5["ConvertedCompYearly"] = pd.to_numeric(df5["ConvertedCompYearly"], errors="coerce")
df5 = df5.dropna(subset=["ConvertedCompYearly", "DevType"])
df5 = df5[df5["ConvertedCompYearly"] < df5["ConvertedCompYearly"].quantile(0.99)]
plt.figure(figsize=(10,6))
sns.boxplot(data=df5, x="DevType", y="ConvertedCompYearly", color='blue')
plt.title("Yearly Converted Compensation for Top 5 Developer Types")
plt.show()

**2. Box Plot of `CompTotal` for the Top 5 Countries:**


Analyze compensation across respondents from the top 5 countries.


In [ ]:
# your code goes here
QUERY6 = """
SELECT CompTotal, Country
FROM main
WHERE CompTotal IS NOT NULL
  AND Country IN (
      SELECT Country
      FROM main
      GROUP BY Country
      ORDER BY COUNT(*) DESC
      LIMIT 5
  )
"""
df6 = pd.read_sql_query(QUERY6, conn)
df6["CompTotal"] = pd.to_numeric(df6["CompTotal"], errors="coerce")
df6 = df6.dropna(subset=["CompTotal", "Country"])
df6 = df6[df6["CompTotal"] < df6["CompTotal"].quantile(0.99)]
plt.figure(figsize=(10,6))
sns.boxplot(data=df6, x="Country", y="CompTotal", color='blue')
plt.title("CompTotal for Top 5 Countries")
plt.show()

### Task 4: Visualizing Comparison of Data


**1. Box Plot of CompTotal Across Employment Types:**


Analyze compensation for different employment types.


In [ ]:
# your code goes here
QUERY7 = """
SELECT CompTotal, Employment
FROM main
WHERE CompTotal IS NOT NULL AND Employment IS NOT NULL
  AND Employment IN (
      SELECT Employment
      FROM main
      GROUP BY Employment
      ORDER BY COUNT(*) DESC
      LIMIT 10
  )
"""
df7 = pd.read_sql_query(QUERY7, conn)
df7["CompTotal"] = pd.to_numeric(df7["CompTotal"], errors="coerce")
df7 = df7.dropna(subset=["CompTotal", "Employment"])
df7 = df7[df7["CompTotal"] < df7["CompTotal"].quantile(0.99)]
df7["Employment"] = df7["Employment"].str.replace(";", ", ")
plt.figure(figsize=(12,6))
sns.boxplot(data=df7, x="Employment", y="CompTotal", color='blue')
plt.title("CompTotal Across Top 10 Employment Types")
plt.show()

**2. Box Plot of `YearsCodePro` by Job Satisfaction (`JobSatPoints_6`):**


Examine the distribution of professional coding years by job satisfaction levels.


In [ ]:
# your code goes here
QUERY8 = """
SELECT YearsCodePro, JobSatPoints_6
FROM main
WHERE YearsCodePro IS NOT NULL AND JobSatPoints_6 IS NOT NULL
"""
df8 = pd.read_sql_query(QUERY8, conn)
df8["YearsCodePro"] = pd.to_numeric(df8["YearsCodePro"], errors="coerce")
df8 = df8.dropna(subset=["YearsCodePro", "JobSatPoints_6"])
top10 = df8["JobSatPoints_6"].value_counts().nlargest(10).index
df8_top10 = df8[df8["JobSatPoints_6"].isin(top10)]
plt.figure(figsize=(10,6))
sns.boxplot(data=df8_top10, x="JobSatPoints_6", y="YearsCodePro", color='blue')
plt.title("Years of Professional Coding by Top 10 Job Satisfaction Levels")
plt.show()

### Final Step: Close the Database Connection


After completing the lab, close the connection to the SQLite database:


In [ ]:
conn.close()

## Summary


In this lab, you used box plots to visualize various aspects of the dataset, focusing on:

- Visualize distributions of compensation and age.

- Explore relationships between compensation, job satisfaction, and professional coding experience.

- Analyze data composition across developer roles and countries.

- Compare compensation across employment types and satisfaction levels.

Box plots provided clear insights into the spread, outliers, and central tendencies of various features in the dataset.


## Authors:
Ayushi Jain


### Other Contributors:
- Rav Ahuja
- Lakshmi Holla
- Malika


<!--## Change Log
|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|               
|2024-10-07|1.2|Madhusudan Moole|Reviewed and updated lab|                                                                                      
|2024-10-06|1.0|Raghul Ramesh|Created lab|-->


Copyright © IBM Corporation. All rights reserved.
